# Function calling: AI tool kaise chalata hai?
**AI Explained · Episode 19 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye. Ek **toy model** hai jo tools ka menu padhta hai aur ek call *likhta* hai. Asli model ek LLM hota; hamara toy sirf words milaata hai (aur sawaal ka pehla word argument bana deta hai), taaki function calling ka flow saaf dikhe. Weather aur slot ka data hamne teaching ke liye banaya hai.

In [1]:
TOOLS = [   # model ko dikhne wala menu
  {"name": "get_weather",
   "description": "kisi city ka abhi ka weather",
   "parameters": {"city": "string"}},
  {"name": "book_slot",
   "description": "mock interview ka slot book",
   "parameters": {"day": ["monday", "tuesday",
     "wednesday", "thursday", "friday"]}},
]
names = [t["name"] for t in TOOLS]
print(len(TOOLS), "tools:", names)

2 tools: ['get_weather', 'book_slot']


In [2]:
def toy_model(q, tools=TOOLS):   # sirf likhta hai
    words = set(q.lower().split())
    for t in tools:
        desc = t["description"].split()
        keys = {w for w in desc if len(w) > 4}
        if words & keys:          # menu se match
            arg = list(t["parameters"])[0]
            first = q.split()[0].lower()
            return {"name": t["name"],
                    "args": {arg: first}}
    return "Mere paas live data nahi hai."

def toy_reply(r):   # result padhkar jawab
    if "error" in r:
        return "Pehle ye batao: " + r["error"]
    if "temp_c" in r:
        c = r["city"].title()
        return f"{c} mein abhi {r['temp_c']}°C hai"
    return f"Slot book: {r['day']}, {r['time']}"

In [3]:
Q = "pune mein abhi weather kaisa hai"
print(toy_model(Q, tools=[]))   # bina tools

Mere paas live data nahi hai.


In [4]:
LOG = []          # jo function sach mein chala
call = toy_model(Q)
print(call)
print("chala:", LOG)

{'name': 'get_weather', 'args': {'city': 'pune'}}
chala: []


In [5]:
WEATHER = {"pune": 31, "delhi": 24}  # nakli

def get_weather(city):        # tumhara code
    LOG.append("get_weather")
    if city not in WEATHER:
        return {"error": "ye city data mein nahi"}
    return {"city": city, "temp_c": WEATHER[city]}

def book_slot(day):           # tumhara code
    LOG.append("book_slot")
    return {"day": day, "time": "4 pm"}

In [6]:
SPEC = {t["name"]: t["parameters"] for t in TOOLS}
FUNCS = {"get_weather": get_weather,
         "book_slot": book_slot}

def check(call):             # args sahi hain?
    spec = SPEC[call["name"]]
    for k, v in call["args"].items():
        ok = spec[k]
        if type(ok) is list and v not in ok:
            return f"{ok[0]}-{ok[-1]}, kaunsa {k}?"

def run_tool(call):          # tumhara code chalata
    err = check(call)
    if err:
        return {"error": err}
    fn = FUNCS[call["name"]]
    return fn(**call["args"])

result = run_tool(call)
print(result)
print("chala:", LOG)

{'city': 'pune', 'temp_c': 31}
chala: ['get_weather']


In [7]:
print(toy_reply(result))   # model ki doosri baari

Pune mein abhi 31°C hai


In [8]:
VAGUE = [{"name": "get_weather",
          "description": "data laata hai",
          "parameters": {"city": "string"}}]
print(toy_model(Q, tools=VAGUE))

Mere paas live data nahi hai.


In [9]:
for q in ["friday ko mock interview book karo",
          "kal mock interview book karo"]:
    c = toy_model(q)
    r = run_tool(c)
    print(c["args"], "->", toy_reply(r))
print("chala:", LOG)

{'day': 'friday'} -> Slot book: friday, 4 pm
{'day': 'kal'} -> Pehle ye batao: monday-friday, kaunsa day?
chala: ['get_weather', 'book_slot']


### Asli duniya mein
OpenAI ka *Function calling* guide flow ke paanch steps deta hai: model ko tools ke saath request bhejo, model se tool call aata hai, *"Execute code on the application side"*, tool ka output lekar doosri request, aur model ka final jawab (ya aur tool calls). Aur: *"When the model calls a function, you must execute it and return the result."*

Anthropic: *"The model can't run your code, so every tool call is a round trip"*, aur model tumhara code kabhi nahi dekhta, sirf schema aur result: *"Claude never receives your implementation; it only receives the schema you provided and the result you returned."* Google (Gemini): *"The model doesn't execute the function itself."*

Description sabse zaroori: model tool ko user ke sawaal aur tool ke description se chunta hai. OpenAI ka test: *"Pass the intern test."* Kya ek intern sirf tumhare diye text se function sahi use kar payega? Enums (jaise monday-friday) galat values rokte hain, aur `strict` mode arguments ko schema se bandhta hai. Ek jawab mein zero, ek ya kai calls ho sakti hain.

In [10]:
ANSWER = ("Function calling means I give the "
  "model a menu of tools: a name, a "
  "description and a JSON schema for the "
  "arguments. The model never runs "
  "anything. It replies with a structured "
  "call: which function, which arguments. "
  "My code checks the arguments, runs the "
  "function and sends the result back, "
  "and the model writes the final answer.")
print(ANSWER)

Function calling means I give the model a menu of tools: a name, a description and a JSON schema for the arguments. The model never runs anything. It replies with a structured call: which function, which arguments. My code checks the arguments, runs the function and sends the result back, and the model writes the final answer.


In [11]:
# TUMHARI BAARI: apna tool menu mein jodo
TOOLS.append({"name": "canteen_menu",
  "description": "aaj canteen ka khana",
  "parameters": {"day": "string"}})
print(toy_model("aaj canteen mein kya hai"))

{'name': 'canteen_menu', 'args': {'day': 'aaj'}}
